# 12 — ELT Report Generation

**Objective**: replace `generate_response`'s Phase 8 one-line-per-project placeholder with `src/agents/report_agent.py`'s real implementation — Section 15's full weekly executive report for portfolio-wide requests, Section 17's Answer/Evidence/Trend/Financial Impact/Recommendation/Confidence/Sources format for single-project requests — plus an optional LLM narration pass over the executive summary.

**Dependencies**: `10_langgraph_workflow.ipynb`, `11_historical_analysis.ipynb`.

**On the LLM piece, honestly**: this environment has no `OPENAI_API_KEY` configured, so `report_agent.build_default_chat_model()` (a real `ChatOpenAI`) is implemented but cannot be exercised here — same status as `JiraCloudRESTSource` and `Mem0MemoryStore` in earlier phases. What CAN be verified without real credentials, and is below: the prompt only ever contains already-computed facts (never asks the model to calculate anything), the chain wiring is correct, and a broken/unavailable model degrades gracefully to the deterministic summary rather than breaking the report. All three are proven against LangChain's own `FakeListChatModel` test double, which is a real dependency-injection point, not a mock of internals.

In [1]:
import os
import sys
import shutil
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from datetime import datetime, timezone

from src.connectors.jira_client import build_default_jira_client
from src.connectors.financial_client import CSVFinancialDataSource
from src.services import project_unifier
from src.services.memory_store import FileMemoryStore
from src.graph.nodes import NodeDeps
from src.graph.workflow import build_graph

NOTEBOOK_SNAPSHOT_DIR = PROJECT_ROOT / "data/snapshots"
shutil.rmtree(NOTEBOOK_SNAPSHOT_DIR, ignore_errors=True)

deps = NodeDeps(
    jira_client=build_default_jira_client(),
    financial_source=CSVFinancialDataSource(),
    memory_store=FileMemoryStore(base_dir=NOTEBOOK_SNAPSHOT_DIR),
    mapping=project_unifier.load_project_mapping(),
)
graph = build_graph(deps)
print("Ready.")

Ready.


## Section 15: the full weekly report

In [2]:
result = graph.invoke({
    "user_question": "What is the status of our portfolio?",
    "request_id": "req-001",
    "requested_at": datetime(2026, 9, 15, tzinfo=timezone.utc),
})
print(result["final_answer"])

{"request_id": "req-001", "timestamp": "2026-09-03T03:23:05.113993+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "req-001", "timestamp": "2026-09-03T03:23:05.114243+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
{"request_id": "req-001", "timestamp": "2026-09-03T03:23:05.114844+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "req-001", "timestamp": "2026-09-03T03:23:05.133458+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 270, "sprint_count": 20, "partial_failure": false}
{"request_id": "req-001", "timestamp": "2026-09-03T03:23:05.133596+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 18.71}
{"request_id": "req-001", "timestamp": "2026-09-03T03:23:05.134194+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "req-001", "timestamp": "2026-09-03T03:23:05.134293+00:00", "event": "graph_node_end", 

Every number above traces back to a node built in Phases 3-8 — this notebook adds no new calculations, only the rendering.

## Section 17: single-project chat answer format

`classify_request`'s `project_filter` (Phase 8) now has real behavioral effect on the OUTPUT FORMAT, not just which data gets fetched — a scoped question gets the Answer/Evidence/Trend/... structure, not a truncated version of the weekly report.

In [3]:
chat_result = graph.invoke({
    "user_question": "Why is Phoenix Platform Modernization at risk?",
    "request_id": "req-002",
    "requested_at": datetime(2026, 9, 15, tzinfo=timezone.utc),
})
print(chat_result["final_answer"])

{"request_id": "req-002", "timestamp": "2026-09-03T03:23:05.197030+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:05.197278+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.02}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:05.197743+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:05.201031+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 37, "sprint_count": 3, "partial_failure": false}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:05.201101+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 3.31}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:05.201572+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "req-002", "timestamp": "2026-09-03T03:23:05.201674+00:00", "event": "graph_node_end", "no

## LLM narration: prompt construction and response wiring, proven without real credentials

In [4]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from src.agents import report_agent

# What the model actually receives — every line traces to an already-computed fact.
facts_text = report_agent.build_narration_facts_text(result["unified_projects"], result["risks"])
print("Facts handed to the model:\n")
print(facts_text)

Facts handed to the model:

total_projects=7
GREEN=0 AMBER=2 RED=5 UNKNOWN=0
approved_budget=$360,384.32
actual_spend=$347,611.89
remaining_budget=$-340,966.91
Phoenix Platform Modernization: status=RED reasons=AGED_BLOCKER,DATA_INCONSISTENT,DEPENDENCY_RISK,FORECAST_OVERRUN,HIGH_BUDGET_CONSUMPTION,NEGATIVE_REMAINING_BUDGET,SPEND_AHEAD_OF_PROGRESS
Orca Payments Gateway: status=RED reasons=AGED_BLOCKER,DATA_INCONSISTENT,DEPENDENCY_RISK,HIGH_BUDGET_CONSUMPTION,LOW_SPRINT_COMPLETION,NEGATIVE_REMAINING_BUDGET,SPEND_AHEAD_OF_PROGRESS
Nova Customer Portal: status=RED reasons=AGED_BLOCKER,DATA_INCONSISTENT,DEPENDENCY_RISK,FORECAST_OVERRUN,HIGH_BUDGET_CONSUMPTION,LOW_SPRINT_COMPLETION,NEGATIVE_REMAINING_BUDGET,SPEND_AHEAD_OF_PROGRESS
Titan Infra Automation: status=RED reasons=AGED_BLOCKER,DATA_INCONSISTENT,DEPENDENCY_RISK,FORECAST_OVERRUN,HIGH_BUDGET_CONSUMPTION,LOW_SPRINT_COMPLETION,NEGATIVE_REMAINING_BUDGET,SPEND_AHEAD_OF_PROGRESS
Lynx Data Analytics: status=RED reasons=AGED_BLOCKER,DATA_INCO

In [5]:
fake_model = FakeListChatModel(responses=[
    "Five of seven tracked projects are RED and two are AMBER, with the portfolio roughly $341,000 over available "
    "budget. Phoenix, Orca, Nova, Titan, and Lynx all need ELT-level intervention this week; Quasar and Helios "
    "are AMBER solely due to unresolved cross-system mapping gaps, not confirmed project health issues."
])

narrated_deps = NodeDeps(
    jira_client=deps.jira_client, financial_source=deps.financial_source,
    memory_store=deps.memory_store, mapping=deps.mapping, chat_model=fake_model,
)
narrated_graph = build_graph(narrated_deps)
narrated_result = narrated_graph.invoke({
    "user_question": "portfolio status",
    "request_id": "req-003",
    "requested_at": datetime(2026, 9, 15, tzinfo=timezone.utc),
})

summary_section = narrated_result["final_answer"].split("## Executive Summary")[1].split("## Project-Level RAG")[0]
print(summary_section)

{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.338744+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.338872+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.339323+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}


{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.358547+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 270, "sprint_count": 20, "partial_failure": false}
{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.358679+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 19.32}
{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.359262+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.359348+00:00", "event": "graph_node_end", "node": "validate_delivery_data", "latency_ms": 0.05}
{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.359757+00:00", "event": "graph_node_start", "node": "fetch_financial_data"}
{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.360020+00:00", "event": "records_retrieved", "source": "Finance", "record_count": 6, "partial_failure": false}
{"request_id": "req-003", "timestamp": "2026-09-03T03:23

{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.444883+00:00", "event": "graph_node_end", "node": "validate_findings", "latency_ms": 33.84}
{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.445594+00:00", "event": "graph_node_start", "node": "generate_response"}


{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.557210+00:00", "event": "graph_node_end", "node": "generate_response", "latency_ms": 111.57}


{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.558135+00:00", "event": "graph_node_start", "node": "persist_snapshot"}
{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.560761+00:00", "event": "datasource_access", "source": "Memory", "operation": "persist_snapshot", "written_count": 7}
{"request_id": "req-003", "timestamp": "2026-09-03T03:23:05.560823+00:00", "event": "graph_node_end", "node": "persist_snapshot", "latency_ms": 2.64}

Five of seven tracked projects are RED and two are AMBER, with the portfolio roughly $341,000 over available budget. Phoenix, Orca, Nova, Titan, and Lynx all need ELT-level intervention this week; Quasar and Helios are AMBER solely due to unresolved cross-system mapping gaps, not confirmed project health issues.
Confidence: LOW CONFIDENCE




Every other section of the report is byte-for-byte identical whether or not a chat model is configured — narration only ever touches the Executive Summary paragraph.

## Graceful degradation: a broken or unavailable model never breaks the report

In [6]:
from langchain_core.runnables import Runnable

class BrokenChatModel(Runnable):
    def invoke(self, input, config=None, **kwargs):
        raise RuntimeError("simulated LLM outage (e.g. no OPENAI_API_KEY configured)")

broken_deps = NodeDeps(
    jira_client=deps.jira_client, financial_source=deps.financial_source,
    memory_store=deps.memory_store, mapping=deps.mapping, chat_model=BrokenChatModel(),
)
broken_graph = build_graph(broken_deps)
broken_result = broken_graph.invoke({
    "user_question": "portfolio status", "request_id": "req-004",
    "requested_at": datetime(2026, 9, 15, tzinfo=timezone.utc),
})
summary = broken_result["final_answer"].split("## Executive Summary")[1].split("## Project-Level RAG")[0]
print(summary)
print("(deterministic summary — the report was produced despite the simulated outage)")

{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.609088+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.609222+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.609705+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.641727+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 270, "sprint_count": 20, "partial_failure": false}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.641798+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 32.04}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.642458+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.642549+00:00", "event": "graph_node_end", 

{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.673275+00:00", "event": "graph_node_end", "node": "validate_financial_data", "latency_ms": 29.56}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.674006+00:00", "event": "graph_node_start", "node": "unify_projects"}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.680893+00:00", "event": "graph_node_end", "node": "unify_projects", "latency_ms": 6.84}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.681451+00:00", "event": "graph_node_start", "node": "retrieve_historical_memory"}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.682227+00:00", "event": "memory_retrieved", "project_count": 7, "snapshot_count": 7}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.682271+00:00", "event": "graph_node_end", "node": "retrieve_historical_memory", "latency_ms": 0.78}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.682693+00:00", "event": "graph_node_start", "node": "calc

{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.788247+00:00", "event": "graph_node_end", "node": "generate_response", "latency_ms": 68.25}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.788929+00:00", "event": "graph_node_start", "node": "persist_snapshot"}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.791347+00:00", "event": "datasource_access", "source": "Memory", "operation": "persist_snapshot", "written_count": 7}
{"request_id": "req-004", "timestamp": "2026-09-03T03:23:05.791410+00:00", "event": "graph_node_end", "node": "persist_snapshot", "latency_ms": 2.44}

7 tracked project(s) — 0 GREEN, 2 AMBER, 5 RED, 0 UNKNOWN.
Portfolio budget: $360,384.32  |  Spend: $347,611.89  |  Available: $-340,966.91
Projects requiring executive attention: Phoenix Platform Modernization, Orca Payments Gateway, Nova Customer Portal, Titan Infra Automation, Lynx Data Analytics, Quasar Self-Service Analytics, Helios Compliance Program
Confidence: LOW CONFIDENCE



## Week-over-week and persistent blockers, in the rendered report

A second run shows real movement in the report text itself, not just in the underlying `ProjectSnapshot` objects (already proven in notebook 11).

In [7]:
second_run = graph.invoke({
    "user_question": "What is the status of our portfolio?",
    "request_id": "req-005",
    "requested_at": datetime(2026, 9, 22, tzinfo=timezone.utc),
})
wow_section = second_run["final_answer"].split("## Week-over-Week Changes")[1].split("## Persistent Blockers")[0]
blockers_section = second_run["final_answer"].split("## Persistent Blockers")[1].split("## Financial Watchlist")[0]
print(wow_section)
print(blockers_section)

{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.798319+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.798543+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.798906+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.873511+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 270, "sprint_count": 20, "partial_failure": false}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.873585+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 74.64}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.874159+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.874246+00:00", "event": "graph_node_end", 

{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.905642+00:00", "event": "graph_node_end", "node": "validate_financial_data", "latency_ms": 30.34}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.906282+00:00", "event": "graph_node_start", "node": "unify_projects"}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.912600+00:00", "event": "graph_node_end", "node": "unify_projects", "latency_ms": 6.28}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.913082+00:00", "event": "graph_node_start", "node": "retrieve_historical_memory"}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.913897+00:00", "event": "memory_retrieved", "project_count": 7, "snapshot_count": 7}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.913939+00:00", "event": "graph_node_end", "node": "retrieve_historical_memory", "latency_ms": 0.82}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:05.914320+00:00", "event": "graph_node_start", "node": "calc

{"request_id": "req-005", "timestamp": "2026-09-03T03:23:06.020059+00:00", "event": "graph_node_end", "node": "generate_response", "latency_ms": 68.91}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:06.020820+00:00", "event": "graph_node_start", "node": "persist_snapshot"}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:06.023336+00:00", "event": "datasource_access", "source": "Memory", "operation": "persist_snapshot", "written_count": 7}
{"request_id": "req-005", "timestamp": "2026-09-03T03:23:06.023419+00:00", "event": "graph_node_end", "node": "persist_snapshot", "latency_ms": 2.54}

Improved: none
Deteriorated: none
No material change: Phoenix Platform Modernization, Orca Payments Gateway, Nova Customer Portal, Titan Infra Automation, Lynx Data Analytics, Quasar Self-Service Analytics, Helios Compliance Program
No prior data (baseline): none



| Issue | Project | Owner | Days Blocked | Sprints Blocked |
|---|---|---|---|---|
| PHX-15 | Phoenix Platform Moderniza

Correctly `STABLE` across the board (this dataset's source systems haven't moved between the two runs — see notebook 11's Part A for the same honest result), and the Persistent Blockers table lists every issue that's been blocked in both of this project's two stored snapshots, with real owner names and blocker ages pulled straight from Jira.

## Validation checks

- [x] Portfolio-wide requests produce Section 15's full 8-section report format
- [x] Scoped (single-project) requests produce Section 17's Answer/Evidence/Trend/Financial Impact/Recommendation/Confidence/Sources format instead
- [x] "Projects Requiring Decisions" and "Executive Actions" show exactly one merged entry per project, not one per risk category (a real duplication bug caught and fixed while building this phase)
- [x] The LLM narration prompt contains only already-computed facts — verified by printing exactly what's sent
- [x] Narration correctly replaces only the Executive Summary paragraph; every other section is identical with or without a chat model
- [x] A broken chat model degrades to the deterministic summary rather than breaking the report
- [x] Week-over-week and persistent-blocker sections reflect real, not fabricated, run-over-run state

## Testing

`tests/test_report_agent.py` (21 tests) covers the deterministic renderers and the narration prompt/fallback logic directly; `tests/test_graph.py` adds 5 more covering the same behavior through full graph invocations.

## Next step

Phase 10: Streamlit dashboard — surfacing `render_weekly_report`/`render_chat_answer`'s output (and the structured data behind it) in the executive-facing UI described in Section 16.